## 15 - Auditoria e correção Hough-only no ASTA (v4)
Este notebook corrige uma divergência de implementação descoberta depois do evento
ASTA v3. O padrão vermelho observado tem periodicidade igual ao stride de 384 px dos tiles.
A auditoria do código encontrou duas diferenças em relação ao baseline sintético congelado:
1. minLineLength usou 512 × 0,25 = 128 px, em vez da diagonal congelada
   hypot(512,512) × 0,25 = 181 px;
2. o NMS exato aplicado depois de HoughLinesP no sintético não foi reproduzido por tile.
O evento v3 permanece imutável. Este notebook não carrega nem executa YOLO, não busca
parâmetros e não altera Canny, limiar Hough, overlap, normalização, merging ou métricas.
Primeiro executa um diagnóstico Hough-only em um frame já observado; os 178 frames só são
liberados após revisão explícita e publicação de um contrato corretivo versionado.

In [1]:
%pip install -q opencv-python-headless pillow pandas tqdm scikit-image

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm
from PIL import Image, ImageDraw, ImageFont
from skimage.morphology import skeletonize
import gc, hashlib, json, math, os, shutil, time, uuid
import cv2
import numpy as np
import pandas as pd

Image.MAX_IMAGE_PIXELS=200_000_000
drive.mount('/content/drive',force_remount=True)

BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
ASTA_RAW=BACKUP_DIR/'data/raw/asta_real'
AUDIT_MANIFEST=BACKUP_DIR/'experiments/asta_external_v3/audit/final/manifest_auditoria_asta_v3.csv'
PROTOCOL_PATH=BACKUP_DIR/'experiments/asta_external_v3/protocol/final/contrato_protocolo_asta_v3.json'
HOUGH_CONFIG_PATH=BACKUP_DIR/'data/synthetic_runs/synthetic_v3_c41d5091de56/hough_v3/config_congelada_v3.json'
OLD_EVENT_ID='4edef981-5590-4330-8dba-bd4d14dbcdda'
OLD_EVENT=BACKUP_DIR/'experiments/asta_external_v3/inference'/f'evento_{OLD_EVENT_ID}'
OLD_CONS=OLD_EVENT/'consolidado'
CORR_ROOT=BACKUP_DIR/'experiments/hough_asta_correction_v4'
DIAG_DIR=CORR_ROOT/'diagnostic'
CONTRACT_DIR=CORR_ROOT/'contract'
EVENTS_DIR=CORR_ROOT/'events'
LOCK_PATH=CORR_ROOT/'TRAVA_EVENTO_HOUGH_ASTA_V4.json'
LOCAL=Path('/content/hough_asta_correction_v4');STAGING=LOCAL/'staging'
for p in [CORR_ROOT,DIAG_DIR,CONTRACT_DIR,EVENTS_DIR,LOCAL,STAGING]:p.mkdir(parents=True,exist_ok=True)

EXPECTED_PROTOCOL='dc4c58381b1e3e7d2a24f7fdac8fc59276e53b9d900853c1191a497319e862bc'
EXPECTED_MANIFEST='91bfa07ebffceaa46271a254d3862348ce2b242a06ed44fc4c07efeeb58722ca'
EXPECTED_HOUGH_CONFIG='14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5'
EXPECTED_OLD_RESULT='eef0f468754b195984a4888cc8c8439f12b5e9db3af94e875082272503542f7c'
DIAGNOSTIC_ID='ML1_20220525_195141_red'

def sha256_arquivo(caminho,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as f:
        for bloco in iter(lambda:f.read(chunk),b''):h.update(bloco)
    return h.hexdigest()
def sha256_canonico(objeto):
    texto=json.dumps(objeto,sort_keys=True,separators=(',',':'),ensure_ascii=False)
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()
def verificar_canonico(registro,campo):
    copia=dict(registro);observado=copia.pop(campo)
    if sha256_canonico(copia)!=observado:raise RuntimeError(f'Hash canônico divergente: {campo}')
    return observado
def gravar_json_atomico(caminho,objeto):
    caminho=Path(caminho);caminho.parent.mkdir(parents=True,exist_ok=True)
    tmp=caminho.with_name(caminho.name+'.tmp')
    tmp.write_text(json.dumps(objeto,indent=2,ensure_ascii=False),encoding='utf-8');os.replace(tmp,caminho)
def copiar_com_hash(origem,destino,esperado,tentativas=3):
    ultimo=None
    for tentativa in range(1,tentativas+1):
        try:
            if destino.exists():destino.unlink()
            shutil.copy2(origem,destino)
            if sha256_arquivo(destino)!=esperado:raise IOError('hash divergente')
            return
        except (OSError,IOError) as exc:
            ultimo=exc
            if destino.exists():destino.unlink()
            if tentativa<tentativas:time.sleep(2**tentativa)
    raise IOError(f'Falha de cópia após {tentativas} tentativas: {ultimo}')

print('Notebook 15 preparado em CPU. YOLO não será carregado nem executado.')

Mounted at /content/drive
Notebook 15 preparado em CPU. YOLO não será carregado nem executado.


## 1. Gate e contrato técnico candidato — sem inferência
Confirma o evento v3, o protocolo, o manifest e a configuração Hough congelada. A correção
é derivada diretamente do código do Notebook 06; nenhum valor é escolhido pelo ASTA.

In [2]:
assert sha256_arquivo(AUDIT_MANIFEST)==EXPECTED_MANIFEST
manifest=pd.read_csv(AUDIT_MANIFEST).sort_values('id_asta').reset_index(drop=True)
assert len(manifest)==178 and manifest.id_asta.nunique()==178 and DIAGNOSTIC_ID in set(manifest.id_asta)
protocolo=json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))
assert verificar_canonico(protocolo,'sha256_contrato_protocolo_asta')==EXPECTED_PROTOCOL
hough_config=json.loads(HOUGH_CONFIG_PATH.read_text(encoding='utf-8'))
assert hough_config['sha256_config']==EXPECTED_HOUGH_CONFIG
old_resumo=json.loads((OLD_CONS/'resumo_final_evento_asta_v3.json').read_text(encoding='utf-8'))
assert verificar_canonico(old_resumo,'sha256_resultado_final')==EXPECTED_OLD_RESULT
assert len(list((OLD_EVENT/'checkpoints/yolo').glob('*.json')))==178
assert len(list((OLD_EVENT/'checkpoints/hough').glob('*.json')))==178

cfg_asta=protocolo['configuracao'];d=hough_config['detector']
TILE=int(cfg_asta['tiles']['tamanho']);STRIDE=int(cfg_asta['tiles']['stride'])
STARTS_X=[i*STRIDE for i in range(int(cfg_asta['tiles']['grid_x']))]
STARTS_Y=[i*STRIDE for i in range(int(cfg_asta['tiles']['grid_y']))]
MIN_LENGTH_SINTETICO=int(math.hypot(TILE,TILE)*float(d['min_length_frac']))
MIN_LENGTH_V3=int(round(TILE*float(d['min_length_frac'])))
assert (TILE,STRIDE,len(STARTS_X),len(STARTS_Y))==(512,384,28,28)
assert MIN_LENGTH_V3==128 and MIN_LENGTH_SINTETICO==181

CONFIG_CORRECAO={
    'protocolo':'hough_asta_equivalence_correction_v4_1','versao':'4.1',
    'base_evento_v3':OLD_EVENT_ID,'sha256_resultado_v3':EXPECTED_OLD_RESULT,
    'sha256_contrato_asta':EXPECTED_PROTOCOL,'sha256_manifest_asta':EXPECTED_MANIFEST,
    'sha256_hough_config':EXPECTED_HOUGH_CONFIG,
    'detector':d,
    'correcoes':{
        'min_line_length_formula':'int(hypot(tile_height,tile_width)*min_length_frac)',
        'min_line_length_px':MIN_LENGTH_SINTETICO,
        'nms_por_tile':{'angulo_max_graus':3.0,'dist_max_px':12.0,'cobertura_min':0.30,
            'origem':'funcao nms_linhas congelada no Notebook 06'},
        'costura_tiles':{'metodo':'particao_proprietaria_por_meio_da_sobreposicao',
            'regra':'cada pixel global pertence exatamente a um tile; segmentos locais sao recortados ao dominio proprietario antes da uniao binaria',
            'origem':'geometria congelada tile=512, stride=384, overlap=128',
            'merging_global_aplicado':False}},
    'inalterado':{'tile':512,'stride':384,'overlap':128,'grid':'28x28',
        'normalizacao':'percentis exatos uint8 1.0-99.8 por frame',
        'canny':[int(d['canny_lo']),int(d['canny_hi'])],
        'hough_threshold':int(d['hough_threshold']),'max_gap':int(d['max_gap']),
        'tolerancia_centerline_px':14.0},
    'substituido_por_defeito_de_equivalencia':{
        'merging_global_v3':cfg_asta['merging_global'],
        'motivo':'o Notebook 06 nao possui merging transitive tiled; a costura deve apenas reconciliar dominios sobrepostos'},
    'proibicoes':{'busca_parametros':True,'recalibracao_asta':True,'execucao_yolo':True,
        'sobrescrever_evento_v3':True},
    'diagnostico_unico_id':DIAGNOSTIC_ID,'bootstrap_replicacoes':2000,'bootstrap_seed':20260823}
CONFIG_CORRECAO['sha256_config_correcao']=sha256_canonico(CONFIG_CORRECAO)
print(json.dumps({'status':'GATE_CORRECAO_HOUGH_V4_1_APROVADO_SEM_INFERENCIA',
    'diagnostico_id':DIAGNOSTIC_ID,'min_length_v3_px':MIN_LENGTH_V3,
    'min_length_fiel_sintetico_px':MIN_LENGTH_SINTETICO,
    'nms_sintetico_reposto':True,'costura_proprietaria_pre_registrada':True,
    'sha256_config_correcao':CONFIG_CORRECAO['sha256_config_correcao'],
    'yolo_executado':False,'hough_executado':False},indent=2,ensure_ascii=False))

{
  "status": "GATE_CORRECAO_HOUGH_V4_1_APROVADO_SEM_INFERENCIA",
  "diagnostico_id": "ML1_20220525_195141_red",
  "min_length_v3_px": 128,
  "min_length_fiel_sintetico_px": 181,
  "nms_sintetico_reposto": true,
  "costura_proprietaria_pre_registrada": true,
  "sha256_config_correcao": "ea65b9d12715fdfb486609605a5ae58deef43535e9f331652feadddeeed71cd7",
  "yolo_executado": false,
  "hough_executado": false
}


## 2. Implementação fiel e testes de equivalência
O NMS vetorizado abaixo preserva a ordem gulosa do código sintético. Ele é comparado com
uma referência escalar em dados artificiais antes que qualquer pixel ASTA seja lido.

In [3]:
def percentil_histograma_uint8(imagem,q):
    hist=np.bincount(imagem.ravel(),minlength=256);alvo=(imagem.size-1)*float(q)/100
    return int(np.searchsorted(np.cumsum(hist),alvo,side='right'))
def normalizar_frame_uint8(imagem):
    lo=percentil_histograma_uint8(imagem,1.0);hi=percentil_histograma_uint8(imagem,99.8)
    if hi<=lo:hi=min(255,lo+1)
    lut=np.clip((np.arange(256,dtype=np.float32)-lo)*255/(hi-lo),0,255).round().astype(np.uint8)
    norm=lut[imagem];return norm,int(percentil_histograma_uint8(norm,50.0))
def extrair_tile(imagem,x,y,preenchimento):
    h,w=imagem.shape;tile=np.full((TILE,TILE),preenchimento,np.uint8)
    vh=max(0,min(TILE,h-y));vw=max(0,min(TILE,w-x))
    if vh and vw:tile[:vh,:vw]=imagem[y:y+vh,x:x+vw]
    return tile
def parametros_linha(p0,p1):
    p0=np.asarray(p0,float);p1=np.asarray(p1,float);v=p1-p0;l=float(np.linalg.norm(v))
    return (p0+p1)/2,v/max(l,1e-9),l
def erro_angular(d1,d2):return float(np.degrees(np.arccos(np.clip(abs(np.dot(d1,d2)),-1,1))))
def distancia_ponto_linha(p,c,d):
    v=np.asarray(p)-np.asarray(c);return float(np.linalg.norm(v-np.dot(v,d)*d))
def cobertura_projetada(p0a,p1a,p0b,p1b):
    cb,db,lb=parametros_linha(p0b,p1b)
    if lb<=0:return 0.0
    t=sorted([np.dot(np.asarray(p0a)-cb,db),np.dot(np.asarray(p1a)-cb,db)])
    return float(max(0,min(t[1],lb/2)-max(t[0],-lb/2))/lb)
def nms_escalar_referencia(linhas):
    ordenadas=sorted(linhas,key=lambda l:-math.dist(l[0],l[1]));mantidas=[]
    for atual in ordenadas:
        ca,da,_=parametros_linha(*atual);suprimir=False
        for m in mantidas:
            cm,dm,_=parametros_linha(*m)
            if erro_angular(da,dm)>3.0:continue
            if max(distancia_ponto_linha(ca,cm,dm),distancia_ponto_linha(cm,ca,da))>12.0:continue
            if max(cobertura_projetada(*atual,*m),cobertura_projetada(*m,*atual))>=0.30:
                suprimir=True;break
        if not suprimir:mantidas.append(atual)
    return mantidas
def nms_vetorizado_exato(linhas):
    if not linhas:return []
    p1=np.asarray([l[0] for l in linhas],float);p2=np.asarray([l[1] for l in linhas],float)
    vec=p2-p1;compr=np.linalg.norm(vec,axis=1);ordem=np.argsort(-compr,kind='stable')
    centros=(p1+p2)/2;dirs=vec/np.maximum(compr[:,None],1e-9);mantidos=[]
    for idx in ordem.tolist():
        if not mantidos:mantidos.append(idx);continue
        k=np.asarray(mantidos,int);dot=np.abs(dirs[k]@dirs[idx]);ang=np.degrees(np.arccos(np.clip(dot,-1,1)))
        possiveis=k[ang<=3.0]
        suprimir=False
        for j in possiveis.tolist():
            if max(distancia_ponto_linha(centros[idx],centros[j],dirs[j]),
                   distancia_ponto_linha(centros[j],centros[idx],dirs[idx]))>12.0:continue
            atual=(p1[idx],p2[idx]);m=(p1[j],p2[j])
            if max(cobertura_projetada(*atual,*m),cobertura_projetada(*m,*atual))>=0.30:
                suprimir=True;break
        if not suprimir:mantidos.append(idx)
    return [linhas[i] for i in mantidos]

def angulo_segmento(s):return math.degrees(math.atan2(s['y2']-s['y1'],s['x2']-s['x1']))%180
def agregar_componente(segmentos,indices):
    pts=[];pesos=[]
    for i in indices:
        s=segmentos[i];peso=max(float(s['peso']),1e-6)
        pts.extend([[s['x1'],s['y1']],[s['x2'],s['y2']]]);pesos.extend([peso,peso])
    pts=np.asarray(pts,float);w=np.asarray(pesos,float);centro=np.average(pts,axis=0,weights=w)
    d=pts-centro;cov=(d*w[:,None]).T@d/max(w.sum(),1e-9);_,v=np.linalg.eigh(cov);eixo=v[:,-1]
    proj=d@eixo;p1=centro+eixo*proj.min();p2=centro+eixo*proj.max()
    return {'x1':float(p1[0]),'y1':float(p1[1]),'x2':float(p2[0]),'y2':float(p2[1]),
        'comprimento':float(np.linalg.norm(p2-p1)),'fragmentos':len(indices),'confianca_max':0.0}
def unir_segmentos(segmentos):
    n=len(segmentos)
    if n==0:return []
    pai=list(range(n))
    def raiz(i):
        while pai[i]!=i:pai[i]=pai[pai[i]];i=pai[i]
        return i
    def unir(i,j):
        a,b=raiz(i),raiz(j)
        if a!=b:pai[b]=a
    p1=np.asarray([[s['x1'],s['y1']] for s in segmentos],float);p2=np.asarray([[s['x2'],s['y2']] for s in segmentos],float)
    vec=p2-p1;norm=np.linalg.norm(vec,axis=1);u=vec/np.maximum(norm[:,None],1e-9)
    ang=np.degrees(np.arctan2(vec[:,1],vec[:,0]))%180;meios=(p1+p2)/2
    limite=cfg_asta['merging_global'];maxa=float(limite['angulo_max_graus']);maxd=float(limite['distancia_perpendicular_max_px']);maxg=float(limite['gap_axial_max_px'])
    def avaliar(ia,ib,mesmo):
        ia=np.asarray(ia,int);ib=np.asarray(ib,int)
        for ini in range(0,len(ia),128):
            a=ia[ini:ini+128];dif=np.abs(ang[a,None]-ang[ib][None,:])%180;mask=np.minimum(dif,180-dif)<=maxa
            if mesmo:mask&=ib[None,:]>a[:,None]
            ra,rb=np.nonzero(mask)
            if not len(ra):continue
            ai=a[ra];bi=ib[rb];va=u[ai];vb=u[bi].copy();vb[np.einsum('ij,ij->i',va,vb)<0]*=-1
            e=va+vb;e/=np.maximum(np.linalg.norm(e,axis=1)[:,None],1e-9);normal=np.column_stack((-e[:,1],e[:,0]))
            keep=np.abs(np.einsum('ij,ij->i',meios[bi]-meios[ai],normal))<=maxd
            ai=ai[keep];bi=bi[keep];e=e[keep]
            if not len(ai):continue
            a1=np.einsum('ij,ij->i',p1[ai],e);a2=np.einsum('ij,ij->i',p2[ai],e);b1=np.einsum('ij,ij->i',p1[bi],e);b2=np.einsum('ij,ij->i',p2[bi],e)
            gap=np.maximum(0,np.maximum(np.minimum(a1,a2),np.minimum(b1,b2))-np.minimum(np.maximum(a1,a2),np.maximum(b1,b2)))
            for i,j in zip(ai[gap<=maxg].tolist(),bi[gap<=maxg].tolist()):unir(i,j)
    buckets={}
    for i,s in enumerate(segmentos):buckets.setdefault((int(s['tile_x']),int(s['tile_y'])),[]).append(i)
    vistos=set()
    for chave in buckets:
        tx,ty=chave
        for ny in range(ty-1,ty+2):
            for nx in range(tx-1,tx+2):
                outra=(nx,ny)
                if outra not in buckets:continue
                par=(chave,outra) if chave<=outra else (outra,chave)
                if par in vistos:continue
                vistos.add(par);avaliar(buckets[par[0]],buckets[par[1]],par[0]==par[1])
    grupos={}
    for i in range(n):grupos.setdefault(raiz(i),[]).append(i)
    return [agregar_componente(segmentos,x) for x in grupos.values()]

rng=np.random.default_rng(20260830);linhas_teste=[]
for _ in range(80):
    p0=rng.uniform(0,512,2);a=rng.uniform(0,np.pi);l=rng.uniform(100,400);p1=p0+l*np.array([np.cos(a),np.sin(a)])
    linhas_teste.append((tuple(p0),tuple(p1)))
ref=nms_escalar_referencia(linhas_teste);vet=nms_vetorizado_exato(linhas_teste)
canon=lambda xs:[tuple(round(float(v),8) for p in x for v in p) for x in xs]
assert canon(ref)==canon(vet)
linhas_sobrepostas=[((0,0),(300,0)),((10,1),(290,1)),((0,30),(300,30)),((0,0),(0,300))]
assert canon(nms_escalar_referencia(linhas_sobrepostas))==canon(nms_vetorizado_exato(linhas_sobrepostas))
assert len(nms_vetorizado_exato(linhas_sobrepostas))==3
print('NMS vetorizado equivalente ao baseline sintético em teste determinístico.')

NMS vetorizado equivalente ao baseline sintético em teste determinístico.


## 3. Diagnóstico de um frame — Hough-only
Altere somente EXECUTAR_DIAGNOSTICO_HOUGH=True. Esta etapa não executa YOLO e não
publica um resultado final. Ela compara o checkpoint v3 existente com a aplicação fiel do
baseline no frame que revelou a malha.

In [4]:
EXECUTAR_DIAGNOSTICO_HOUGH=False

def limites_propriedade(starts,indice,n):
    inicio=0 if indice==0 else int(round((starts[indice-1]+TILE+starts[indice])/2))
    fim=n if indice==len(starts)-1 else int(round((starts[indice]+TILE+starts[indice+1])/2))
    return inicio,fim
def validar_particao_proprietaria(starts,n):
    intervalos=[limites_propriedade(starts,i,n) for i in range(len(starts))]
    assert intervalos[0][0]==0 and intervalos[-1][1]==n
    assert all(a<b for a,b in intervalos)
    assert all(intervalos[i][1]==intervalos[i+1][0] for i in range(len(intervalos)-1))
    assert sum(b-a for a,b in intervalos)==n
    return intervalos
PROPRIEDADE_X=validar_particao_proprietaria(STARTS_X,10560)
PROPRIEDADE_Y=validar_particao_proprietaria(STARTS_Y,10560)
def recortar_ao_dominio_proprietario(p1,p2,ix,iy):
    esquerda,direita=PROPRIEDADE_X[ix];topo,base=PROPRIEDADE_Y[iy]
    a=(int(round(p1[0])),int(round(p1[1])));b=(int(round(p2[0])),int(round(p2[1])))
    ok,q1,q2=cv2.clipLine((esquerda,topo,direita-esquerda,base-topo),a,b)
    if not ok:return None
    comp=float(math.dist(q1,q2))
    if comp<=0:return None
    return {'x1':float(q1[0]),'y1':float(q1[1]),'x2':float(q2[0]),'y2':float(q2[1]),
        'peso':comp,'comprimento':comp,'confianca':0.0,'tile_x':ix,'tile_y':iy}
def detectar_hough_corrigido(imagem_norm,preenchimento,id_asta):
    segmentos=[];proprietarios=[];raw_total=0;nms_total=0;tempos={'tiles':0.0,'hough_nms':0.0}
    coords=[(ix,iy,x,y) for iy,y in enumerate(STARTS_Y) for ix,x in enumerate(STARTS_X)]
    for ix,iy,x,y in tqdm(coords,desc=f'{id_asta} Hough corrigido',unit='tile',leave=False):
        t=time.perf_counter();tile=extrair_tile(imagem_norm,x,y,preenchimento);tempos['tiles']+=time.perf_counter()-t
        base=cv2.GaussianBlur(tile,(0,0),float(d['pre_blur_sigma'])) if float(d.get('pre_blur_sigma',0))>0 else tile
        t=time.perf_counter();bordas=cv2.Canny(base,int(d['canny_lo']),int(d['canny_hi']))
        seed=int(hashlib.sha256(f'{id_asta}|{ix}|{iy}|hough_v4'.encode()).hexdigest()[:8],16)&0x7fffffff;cv2.setRNGSeed(seed)
        linhas=cv2.HoughLinesP(bordas,1,np.pi/180,threshold=int(d['hough_threshold']),minLineLength=MIN_LENGTH_SINTETICO,maxLineGap=int(d['max_gap']))
        locais=[] if linhas is None else [((int(a),int(b)),(int(c),int(e))) for a,b,c,e in np.asarray(linhas).reshape(-1,4)]
        raw_total+=len(locais);mantidas=nms_vetorizado_exato(locais);nms_total+=len(mantidas)
        for (x1,y1),(x2,y2) in mantidas:
            p1=np.array([x1+x,y1+y],float);p2=np.array([x2+x,y2+y],float);centro=(p1+p2)/2
            if not (0<=centro[0]<10560 and 0<=centro[1]<10560):continue
            comp=float(np.linalg.norm(p2-p1));segmentos.append({'x1':float(p1[0]),'y1':float(p1[1]),'x2':float(p2[0]),'y2':float(p2[1]),'peso':comp,'comprimento':comp,'confianca':0.0,'tile_x':ix,'tile_y':iy})
            recortado=recortar_ao_dominio_proprietario(p1,p2,ix,iy)
            if recortado is not None:proprietarios.append(recortado)
        tempos['hough_nms']+=time.perf_counter()-t
    return segmentos,proprietarios,{'segmentos_houghlinesp':raw_total,'segmentos_pos_nms':nms_total,
        'segmentos_pos_costura_proprietaria':len(proprietarios),**tempos}
def rasterizar(segmentos,shape):
    h,w=shape;canvas=np.zeros((h,w),np.uint8)
    for s in segmentos:
        p1=(int(round(s['x1'])),int(round(s['y1'])));p2=(int(round(s['x2'])),int(round(s['y2'])))
        ok,a,b=cv2.clipLine((0,0,w,h),p1,p2)
        if ok:cv2.line(canvas,a,b,1,1,cv2.LINE_8)
    return canvas
def metricas_canvas(pred,gt,tol=14):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*tol+1,2*tol+1));gd=cv2.dilate(gt.astype(np.uint8),k);pd=cv2.dilate(pred,k)
    pp=int(pred.sum());gp=int(gt.sum());ph=int(((pred>0)&(gd>0)).sum());gh=int(((gt>0)&(pd>0)).sum())
    p=ph/pp if pp else (1.0 if gp==0 else 0.0);r=gh/gp if gp else None;f=2*p*r/(p+r) if r is not None and p+r else (0.0 if gp else None)
    return {'pred_pixels':pp,'gt_pixels':gp,'pred_hits':ph,'gt_hits':gh,'precisao_centerline':float(p),'recall_centerline':None if r is None else float(r),'f1_centerline':None if f is None else float(f)}
def diagnostico_grade(segmentos,pred,band=3):
    pontos=np.asarray([[s[k] for k in ['x1','y1','x2','y2']] for s in segmentos],float).reshape(-1,2) if segmentos else np.empty((0,2))
    dist=lambda v:np.minimum(np.mod(v,STRIDE),STRIDE-np.mod(v,STRIDE))
    alinh=float(np.mean((dist(pontos[:,0])<=band)|(dist(pontos[:,1])<=band))) if len(pontos) else 0.0
    ang=np.asarray([angulo_segmento(s) for s in segmentos]);eixo=float(np.mean((np.minimum(ang,180-ang)<=3)|(np.abs(ang-90)<=3))) if len(ang) else 0.0
    rows=sorted({v for y in STARTS_Y for v in range(max(0,y-band),min(pred.shape[0],y+band+1))})
    cols=sorted({v for x in STARTS_X for v in range(max(0,x-band),min(pred.shape[1],x+band+1))})
    total=int(pred.sum());hits=int(pred[rows,:].sum()+pred[:,cols].sum()-pred[np.ix_(rows,cols)].sum()) if total else 0
    area=1-(1-len(rows)/pred.shape[0])*(1-len(cols)/pred.shape[1]);frac=hits/total if total else 0.0
    return {'segmentos':len(segmentos),'endpoints_proximos_grade_frac':alinh,'segmentos_quase_horiz_vert_frac':eixo,'pixels_em_faixa_grade_frac':frac,'fracao_area_faixa_grade':float(area),'enriquecimento_grade':float(frac/area) if area else None}
def overlay(base,dens,cor):
    alpha=np.clip(np.sqrt(np.maximum(dens,0))*1.8,0,0.85)[...,None];c=np.asarray(cor,float).reshape(1,1,3)
    return np.clip(base*(1-alpha)+c*alpha,0,255).astype(np.uint8)

if not EXECUTAR_DIAGNOSTICO_HOUGH:
    print('Diagnóstico bloqueado. Altere somente EXECUTAR_DIAGNOSTICO_HOUGH=True.')
else:
    linha=manifest.set_index('id_asta').loc[DIAGNOSTIC_ID]
    imgp=STAGING/linha.arquivo_imagem;maskp=STAGING/linha.arquivo_mascara
    copiar_com_hash(ASTA_RAW/linha.arquivo_imagem,imgp,linha.sha256_imagem);copiar_com_hash(ASTA_RAW/linha.arquivo_mascara,maskp,linha.sha256_mascara)
    imagem=cv2.imread(str(imgp),cv2.IMREAD_GRAYSCALE);mascara=cv2.imread(str(maskp),cv2.IMREAD_GRAYSCALE)
    assert imagem.shape==mascara.shape==(10560,10560);norm,med=normalizar_frame_uint8(imagem);gt=skeletonize(mascara>0).astype(np.uint8)
    atual=json.loads((OLD_EVENT/'checkpoints/hough'/f'{DIAGNOSTIC_ID}.json').read_text(encoding='utf-8'))
    assert atual['event_id']==OLD_EVENT_ID and atual['id_asta']==DIAGNOSTIC_ID and verificar_canonico(atual,'sha256_checkpoint')==atual['sha256_checkpoint']
    pred_atual=rasterizar(atual['segmentos'],gt.shape);grid_atual=diagnostico_grade(atual['segmentos'],pred_atual)
    t=time.perf_counter();brutos,proprietarios,stats=detectar_hough_corrigido(norm,med,DIAGNOSTIC_ID)
    t_merge=time.perf_counter();unidos=unir_segmentos(brutos);tempo_merge=time.perf_counter()-t_merge
    pred_merge=rasterizar(unidos,gt.shape);met_merge=metricas_canvas(pred_merge,gt);grid_merge=diagnostico_grade(unidos,pred_merge)
    pred_owner=rasterizar(proprietarios,gt.shape);met_owner=metricas_canvas(pred_owner,gt);grid_owner=diagnostico_grade(proprietarios,pred_owner)
    preview=640;base=cv2.cvtColor(cv2.resize(norm,(preview,preview),interpolation=cv2.INTER_AREA),cv2.COLOR_GRAY2RGB).astype(float)
    dens_gt=cv2.resize(gt.astype(np.float32),(preview,preview),interpolation=cv2.INTER_AREA)
    dens_a=cv2.resize(pred_atual.astype(np.float32),(preview,preview),interpolation=cv2.INTER_AREA)
    dens_m=cv2.resize(pred_merge.astype(np.float32),(preview,preview),interpolation=cv2.INTER_AREA)
    dens_o=cv2.resize(pred_owner.astype(np.float32),(preview,preview),interpolation=cv2.INTER_AREA)
    quadros=[base.astype(np.uint8),overlay(base,dens_gt,(0,255,80)),overlay(base,dens_a,(255,60,40)),
        overlay(base,dens_m,(255,60,40)),overlay(base,dens_o,(255,60,40))]
    canvas=Image.new('RGB',(preview*3,preview*2+110),'white');draw=ImageDraw.Draw(canvas)
    try:fonte=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',20)
    except Exception:fonte=ImageFont.load_default()
    titulos=['Imagem limpa','GT oficial — verde',f'Hough v3 — {len(atual["segmentos"])} segmentos',
        f'Fiel + merging v3 — {len(unidos)} segmentos',f'Fiel + costura proprietária — {len(proprietarios)} segmentos']
    for i,(q,tit) in enumerate(zip(quadros,titulos)):
        x=(i%3)*preview;y=70+(i//3)*preview;canvas.paste(Image.fromarray(q),(x,y));draw.text((x+10,45+(i//3)*preview),tit,fill='black',font=fonte)
    draw.text((10,10),f'{DIAGNOSTIC_ID} | auditoria de costura, sem YOLO e sem busca de parâmetros',fill='black',font=fonte)
    imagem_saida=DIAG_DIR/'comparacao_hough_v3_merge_costura_proprietaria_um_frame.png';canvas.save(imagem_saida)
    rel={'protocolo':'hough_asta_diagnostic_v4_1','status':'DIAGNOSTICO_COSTURA_HOUGH_UM_FRAME_CONCLUIDO','id_asta':DIAGNOSTIC_ID,
        'sha256_config_correcao':CONFIG_CORRECAO['sha256_config_correcao'],'hough_v3':{'metricas':atual['metricas'],'grade':grid_atual,'segmentos_apos_merging':len(atual['segmentos'])},
        'hough_fiel_com_merging_v3':{'metricas':met_merge,'grade':grid_merge,'segmentos_houghlinesp':stats['segmentos_houghlinesp'],
            'segmentos_pos_nms':stats['segmentos_pos_nms'],'segmentos_apos_merging':len(unidos)},
        'hough_fiel_com_costura_proprietaria':{'metricas':met_owner,'grade':grid_owner,
            'segmentos_houghlinesp':stats['segmentos_houghlinesp'],'segmentos_pos_nms':stats['segmentos_pos_nms'],
            'segmentos_pos_costura_proprietaria':len(proprietarios),'cobertura_particao_exata':True,'merging_global_aplicado':False},
        'tempos_segundos':{'total_diagnostico':time.perf_counter()-t,'merging_v3_comparativo':tempo_merge},'yolo_executado':False,'busca_parametros':False,
        'sha256_imagem_diagnostico':sha256_arquivo(imagem_saida)}
    rel['sha256_relatorio_diagnostico']=sha256_canonico(rel);gravar_json_atomico(DIAG_DIR/'relatorio_diagnostico_hough_v4_1.json',rel)
    print(json.dumps(rel,indent=2,ensure_ascii=False));print('Envie o relatório e comparacao_hough_v3_merge_costura_proprietaria_um_frame.png antes da seção 4.')
    for p in [imgp,maskp]:
        if p.exists():p.unlink()
    del imagem,mascara,norm,gt,pred_atual,pred_merge,pred_owner,base,dens_gt,dens_a,dens_m,dens_o,canvas;gc.collect()

Diagnóstico bloqueado. Altere somente EXECUTAR_DIAGNOSTICO_HOUGH=True.


## 4. Publicação do contrato corretivo
Só habilite após revisar o diagnóstico. A aprovação confirma correção de equivalência,
não escolha do melhor resultado.

In [6]:
APROVAR_CORRECAO_HOUGH=False
NOTA_APROVACAO=(
    'A costura proprietária eliminou a concentração periódica no grid de tiles, '
    'com enriquecimento 0,894 e cobertura exata do frame. O resultado permaneceu '
    'saturado por falsos positivos; esse desempenho ruim foi preservado sem busca '
    'de parâmetros ou recalibração no ASTA. A correção é pós-teste, limitada à '
    'equivalência de implementação, e não uma otimização do Hough.'
)

CONTRACT_PATH=CONTRACT_DIR/'contrato_correcao_hough_asta_v4.json'
if not APROVAR_CORRECAO_HOUGH:
    print('Contrato corretivo bloqueado. Revise primeiro o diagnóstico de um frame.')
else:
    if NOTA_APROVACAO.strip() in {'','Unknown / to be confirmed'}:raise RuntimeError('Registre a justificativa de aprovação.')
    diag=json.loads((DIAG_DIR/'relatorio_diagnostico_hough_v4_1.json').read_text(encoding='utf-8'))
    assert verificar_canonico(diag,'sha256_relatorio_diagnostico')==diag['sha256_relatorio_diagnostico']
    assert diag['sha256_config_correcao']==CONFIG_CORRECAO['sha256_config_correcao'] and not diag['yolo_executado'] and not diag['busca_parametros']
    assert diag['status']=='DIAGNOSTICO_COSTURA_HOUGH_UM_FRAME_CONCLUIDO'
    assert diag['hough_fiel_com_costura_proprietaria']['cobertura_particao_exata']
    assert not diag['hough_fiel_com_costura_proprietaria']['merging_global_aplicado']
    contrato={'protocolo':'hough_asta_correction_contract_v4','status':'APROVADO_PARA_EVENTO_HOUGH_ONLY',
        'aprovado_em_utc':datetime.now(timezone.utc).isoformat(),'configuracao':CONFIG_CORRECAO,
        'sha256_relatorio_diagnostico':diag['sha256_relatorio_diagnostico'],'nota_aprovacao':NOTA_APROVACAO.strip(),
        'evento_v3_preservado':True,'yolo_reexecucao_proibida':True,'ajuste_por_resultado_asta':False}
    contrato['sha256_contrato_correcao']=sha256_canonico(contrato)
    if CONTRACT_PATH.exists():
        existente=json.loads(CONTRACT_PATH.read_text(encoding='utf-8'));assert existente==contrato
    else:gravar_json_atomico(CONTRACT_PATH,contrato)
    print(json.dumps(contrato,indent=2,ensure_ascii=False))

Contrato corretivo bloqueado. Revise primeiro o diagnóstico de um frame.


## 5. Evento corretivo completo — 178 frames, somente Hough
Execute somente após compartilhar o contrato aprovado. O evento é retomável por frame,
grava em diretório novo e nunca modifica o evento v3 ou os checkpoints YOLO. O Hough é
aplicado fielmente por tile e costurado pela partição proprietária pré-registrada; o merging
global transitivo v3 não é utilizado.

In [ ]:
EXECUTAR_EVENTO_HOUGH_V4=True
RETOMAR_EVENTO_HOUGH_V4=False

def resumo_metodo(registros):
    pos=[r for r in registros if r['gt_positivo']];neg=[r for r in registros if not r['gt_positivo']]
    vals=[r['metricas']['f1_centerline'] for r in pos]
    pp=sum(r['metricas']['pred_pixels'] for r in registros);ph=sum(r['metricas']['pred_hits'] for r in registros)
    gp=sum(r['metricas']['gt_pixels'] for r in pos);gh=sum(r['metricas']['gt_hits'] for r in pos)
    p=ph/pp if pp else 0.0;r=gh/gp if gp else 0.0;f=2*p*r/(p+r) if p+r else 0.0
    mp=10560*10560/1e6;fp_neg=sum(r['metricas']['pred_pixels'] for r in neg)/max(len(neg)*mp,1e-9)
    return {'frames':len(registros),'frames_positivos':len(pos),'frames_negativos':len(neg),'f1_centerline_macro_frames_positivos':float(np.mean(vals)),
        'precisao_centerline_pooled':float(p),'recall_centerline_pooled':float(r),'f1_centerline_pooled':float(f),
        'comprimento_fp_por_megapixel_negativo':float(fp_neg),'taxa_frames_negativos_com_fp':float(np.mean([x['metricas']['pred_pixels']>0 for x in neg]))}
def checkpoint_valido(path,event_id,id_asta):
    if not path.is_file():return None
    r=json.loads(path.read_text(encoding='utf-8'))
    if r.get('event_id')!=event_id or r.get('id_asta')!=id_asta or r.get('sha256_config_correcao')!=CONFIG_CORRECAO['sha256_config_correcao']:return None
    verificar_canonico(r,'sha256_checkpoint');return r

if not EXECUTAR_EVENTO_HOUGH_V4:
    print('Evento completo bloqueado. Compartilhe primeiro o contrato corretivo.')
else:
    if not CONTRACT_PATH.is_file():raise RuntimeError('Contrato corretivo ausente.')
    contrato=json.loads(CONTRACT_PATH.read_text(encoding='utf-8'));verificar_canonico(contrato,'sha256_contrato_correcao')
    assert contrato['status']=='APROVADO_PARA_EVENTO_HOUGH_ONLY' and contrato['configuracao']['sha256_config_correcao']==CONFIG_CORRECAO['sha256_config_correcao']
    if LOCK_PATH.is_file():
        lock=json.loads(LOCK_PATH.read_text(encoding='utf-8'))
        if lock['status']=='concluido':raise RuntimeError('Evento Hough v4 já concluído; reexecução proibida.')
        if not RETOMAR_EVENTO_HOUGH_V4:raise RuntimeError('Evento incompleto existe. Use RETOMAR_EVENTO_HOUGH_V4=True.')
        assert lock['sha256_config_correcao']==CONFIG_CORRECAO['sha256_config_correcao'];event_id=lock['event_id']
    else:
        if RETOMAR_EVENTO_HOUGH_V4:raise RuntimeError('Não existe evento para retomar.')
        event_id=str(uuid.uuid4());lock={'protocolo':'hough_asta_correction_event_v4','event_id':event_id,'status':'em_andamento',
            'iniciado_em_utc':datetime.now(timezone.utc).isoformat(),'sha256_config_correcao':CONFIG_CORRECAO['sha256_config_correcao'],'frames_concluidos':0}
        gravar_json_atomico(LOCK_PATH,lock)
    event_dir=EVENTS_DIR/f'evento_{event_id}';check_dir=event_dir/'checkpoints';cons=event_dir/'consolidado'
    check_dir.mkdir(parents=True,exist_ok=True);cons.mkdir(parents=True,exist_ok=True)
    for linha in tqdm(list(manifest.itertuples()),desc='Frames Hough ASTA v4',unit='frame'):
        cp=check_dir/f'{linha.id_asta}.json';existente=checkpoint_valido(cp,event_id,linha.id_asta)
        if existente is not None:continue
        imgp=STAGING/linha.arquivo_imagem;maskp=STAGING/linha.arquivo_mascara
        copiar_com_hash(ASTA_RAW/linha.arquivo_imagem,imgp,linha.sha256_imagem);copiar_com_hash(ASTA_RAW/linha.arquivo_mascara,maskp,linha.sha256_mascara)
        imagem=cv2.imread(str(imgp),cv2.IMREAD_GRAYSCALE);mascara=cv2.imread(str(maskp),cv2.IMREAD_GRAYSCALE);assert imagem.shape==mascara.shape==(10560,10560)
        t0=time.perf_counter();norm,med=normalizar_frame_uint8(imagem);tempo_pre=time.perf_counter()-t0
        brutos,proprietarios,stats=detectar_hough_corrigido(norm,med,linha.id_asta)
        gt=skeletonize(mascara>0).astype(np.uint8);pred=rasterizar(proprietarios,gt.shape);met=metricas_canvas(pred,gt);grade=diagnostico_grade(proprietarios,pred)
        reg={'protocolo':'hough_asta_corrected_frame_v4','event_id':event_id,'sha256_config_correcao':CONFIG_CORRECAO['sha256_config_correcao'],
            'id_asta':linha.id_asta,'sha256_imagem':linha.sha256_imagem,'sha256_mascara':linha.sha256_mascara,'gt_positivo':bool(int(linha.mascara_pixels_positivos)>0),
            'segmentos_houghlinesp':stats['segmentos_houghlinesp'],'segmentos_pos_nms':stats['segmentos_pos_nms'],
            'segmentos_pos_costura_proprietaria':len(proprietarios),'segmentos':proprietarios,'metricas':met,'diagnostico_grade':grade,
            'costura':'particao_proprietaria_sem_merging_global','tempos_segundos':{'preprocessamento':tempo_pre+stats['tiles'],
            'inferencia_hough_nms_costura':stats['hough_nms'],'merging':0.0,
            'total_frame':tempo_pre+stats['tiles']+stats['hough_nms']},'concluido':True}
        reg['sha256_checkpoint']=sha256_canonico(reg);gravar_json_atomico(cp,reg)
        lock['frames_concluidos']=len(list(check_dir.glob('*.json')));gravar_json_atomico(LOCK_PATH,lock)
        for p in [imgp,maskp]:
            if p.exists():p.unlink()
        del imagem,mascara,norm,gt,pred,brutos,proprietarios;gc.collect()
    registros=[checkpoint_valido(check_dir/f'{i}.json',event_id,i) for i in manifest.id_asta.astype(str)]
    assert all(x is not None for x in registros) and len(registros)==178
    resumo_h=resumo_metodo(registros)
    yolo=[]
    for id_asta in manifest.id_asta.astype(str):
        r=json.loads((OLD_EVENT/'checkpoints/yolo'/f'{id_asta}.json').read_text(encoding='utf-8'));assert r['event_id']==OLD_EVENT_ID and r['id_asta']==id_asta;verificar_canonico(r,'sha256_checkpoint');yolo.append(r)
    positivos=[i for i,r in enumerate(registros) if r['gt_positivo']];rng=np.random.default_rng(20260823);boot=[]
    yh=np.asarray([yolo[i]['metricas']['f1_centerline'] for i in positivos],float);hh=np.asarray([registros[i]['metricas']['f1_centerline'] for i in positivos],float)
    for _ in tqdm(range(2000),desc='Bootstrap Hough corrigido',unit='rep'):
        idx=rng.integers(0,len(positivos),len(positivos));my=float(yh[idx].mean());mh=float(hh[idx].mean());boot.append((my,mh,my-mh))
    b=np.asarray(boot);ic={'f1_yolo':[float(x) for x in np.percentile(b[:,0],[2.5,97.5])],
        'f1_hough_corrigido':[float(x) for x in np.percentile(b[:,1],[2.5,97.5])],
        'delta_yolo_menos_hough':[float(x) for x in np.percentile(b[:,2],[2.5,97.5])]}
    resumo={'protocolo':'hough_asta_correction_event_v4','status':'EVENTO_HOUGH_ASTA_V4_CONCLUIDO','event_id':event_id,
        'sha256_config_correcao':CONFIG_CORRECAO['sha256_config_correcao'],'evento_v3_preservado':OLD_EVENT_ID,
        'resultado_hough_corrigido':resumo_h,'resultado_yolo_reutilizado_sem_inferencia':old_resumo['resultado_primario_yolo'],
        'bootstrap_pareado_frames_positivos':{'replicacoes':2000,'seed':20260823,'n_frames':177,'ic95_percentil':ic},
        'yolo_executado':False,'busca_parametros':False,'recalibracao_asta':False}
    resumo['sha256_resultado_final']=sha256_canonico(resumo);gravar_json_atomico(cons/'resumo_final_hough_asta_v4.json',resumo)
    tabela=pd.DataFrame([{'id_asta':r['id_asta'],**r['metricas'],'segmentos_houghlinesp':r['segmentos_houghlinesp'],
        'segmentos_pos_nms':r['segmentos_pos_nms'],'segmentos_pos_costura_proprietaria':r['segmentos_pos_costura_proprietaria'],
        **r['diagnostico_grade']} for r in registros])
    tabela.to_csv(cons/'metricas_hough_asta_v4.csv',index=False)
    lock.update({'status':'concluido','concluido_em_utc':datetime.now(timezone.utc).isoformat(),'frames_concluidos':178,'sha256_resultado_final':resumo['sha256_resultado_final']});gravar_json_atomico(LOCK_PATH,lock)
    print(json.dumps(resumo,indent=2,ensure_ascii=False));print('EVENTO Hough v4 concluído. YOLO não foi reexecutado; evento v3 preservado.')

Frames Hough ASTA v4:   0%|          | 0/178 [00:00<?, ?frame/s]

ML1_20170902_022840_red Hough corrigido:   0%|          | 0/784 [00:00<?, ?tile/s]

ML1_20171023_002210_red Hough corrigido:   0%|          | 0/784 [00:00<?, ?tile/s]

ML1_20171023_003240_red Hough corrigido:   0%|          | 0/784 [00:00<?, ?tile/s]

ML1_20171023_014550_red Hough corrigido:   0%|          | 0/784 [00:00<?, ?tile/s]

...

ML1_20230110_185805_red Hough corrigido:   0%|          | 0/784 [00:00<?, ?tile/s]

ML1_20230219_185141_red Hough corrigido:   0%|          | 0/784 [00:00<?, ?tile/s]

ML1_20230227_020410_red Hough corrigido:   0%|          | 0/784 [00:00<?, ?tile/s]

Bootstrap Hough corrigido:   0%|          | 0/2000 [00:00<?, ?rep/s]

{
  "protocolo": "hough_asta_correction_event_v4",
  "status": "EVENTO_HOUGH_ASTA_V4_CONCLUIDO",
  "event_id": "7d717aea-eed8-4f4b-ba63-d4b0ca51d29c",
  "sha256_config_correcao": "ea65b9d12715fdfb486609605a5ae58deef43535e9f331652feadddeeed71cd7",
  "evento_v3_preservado": "4edef981-5590-4330-8dba-bd4d14dbcdda",
  "resultado_hough_corrigido": {
    "frames": 178,
    "frames_positivos": 177,
    "frames_negativos": 1,
    "f1_centerline_macro_frames_positivos": 0.004631929293736692,
    "precisao_centerline_pooled": 0.00231256148939947,
    "recall_centerline_pooled": 0.9714565753269572,
    "f1_centerline_pooled": 0.0046141389776839774,
    "comprimento_fp_por_megapixel_negativo": 55116.523903810834,
    "taxa_frames_negativos_com_fp": 1.0
  },
  "resultado_yolo_reutilizado_sem_inferencia": {
    "metodo": "YOLO11n-OBB seed 20260824",
    "frames": 178,
    "frames_positivos": 177,
    "frames_negativos": 1,
    "f1_centerline_macro_frames_positivos": 0.5833671994488407,
    "precisao_

## 6. Verificação pós-evento

In [7]:
if not LOCK_PATH.is_file():
    print('Nenhum evento Hough v4 publicado ainda.')
else:
    lock=json.loads(LOCK_PATH.read_text(encoding='utf-8'))
    if lock['status']!='concluido':
        print(json.dumps({'status':'EVENTO_HOUGH_V4_INCOMPLETO','event_id':lock['event_id'],'frames_concluidos':lock['frames_concluidos']},indent=2))
    else:
        event_dir=EVENTS_DIR/f"evento_{lock['event_id']}";arquivos=list((event_dir/'checkpoints').glob('*.json'));assert len(arquivos)==178
        for p in tqdm(arquivos,desc='Verificando checkpoints Hough v4',unit='frame'):
            r=json.loads(p.read_text(encoding='utf-8'));verificar_canonico(r,'sha256_checkpoint');assert r['event_id']==lock['event_id']
        resumo=json.loads((event_dir/'consolidado/resumo_final_hough_asta_v4.json').read_text(encoding='utf-8'))
        assert verificar_canonico(resumo,'sha256_resultado_final')==lock['sha256_resultado_final']
        print(json.dumps({'status':'BACKUP_EVENTO_HOUGH_ASTA_V4_APROVADO','event_id':lock['event_id'],'frames':178,
            'sha256_resultado_final':lock['sha256_resultado_final'],'evento_v3_preservado':True,'yolo_reexecutado':False},indent=2,ensure_ascii=False))

Verificando checkpoints Hough v4:   0%|          | 0/178 [00:00<?, ?frame/s]

{
  "status": "BACKUP_EVENTO_HOUGH_ASTA_V4_APROVADO",
  "event_id": "7d717aea-eed8-4f4b-ba63-d4b0ca51d29c",
  "frames": 178,
  "sha256_resultado_final": "303502ee091dfa26649df5bef4f61f0cf8d644925fd9487f96b2085772b444d8",
  "evento_v3_preservado": true,
  "yolo_reexecutado": false
}
